# Experiment 6

## 6.1 GloVe (Global Vectors for Word Representation)
Generate word embeddings from text corpus and find most similar words to 'King'.

### (a) Without using any pretrained model (from scratch)

In [3]:
%pip install numpy


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:

import numpy as np
import re
from collections import Counter, defaultdict

# Read corpus
with open("glove_corpus.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

# Tokenize
tokens = re.findall(r'\b[a-z]+\b', text)
vocab = list(dict.fromkeys(tokens))
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print(f"Vocabulary size: {V}, Total tokens: {len(tokens)}")

# Build co-occurrence matrix with distance weighting
window = 2
X = np.zeros((V, V))
for idx, w in enumerate(tokens):
    wi = w2i[w]
    for j in range(max(0, idx - window), min(len(tokens), idx + window + 1)):
        if j == idx:
            continue
        wj = w2i[tokens[j]]
        dist = abs(j - idx)
        X[wi, wj] += 1.0 / dist

# GloVe hyperparameters
D = 50
rng = np.random.default_rng(42)
W = (rng.random((V, D)) - 0.5) / D   # target vectors
C = (rng.random((V, D)) - 0.5) / D   # context vectors
b = np.zeros(V)
bc = np.zeros(V)

xmax, alpha = 10.0, 0.75
def weight_fn(x):
    return (x / xmax) ** alpha if x < xmax else 1.0

epochs = 15
lr = 0.05
for e in range(epochs):
    loss = 0.0
    for i in range(V):
        for j in range(V):
            if X[i, j] == 0:
                continue
            w_ij = weight_fn(X[i, j])
            diff = W[i] @ C[j] + b[i] + bc[j] - np.log(X[i, j])
            grad = w_ij * diff
            loss += 0.5 * w_ij * diff * diff
            W[i] -= lr * grad * C[j]
            C[j] -= lr * grad * W[i].copy()
            b[i] -= lr * grad
            bc[j] -= lr * grad
    print(f"Epoch {e+1}/{epochs} - Loss: {loss:.4f}")

# Final embeddings
embeddings = W + C

# Most similar words to "king"
target = "king"
ti = w2i[target]
v = embeddings[ti]
sims = []
for w in vocab:
    if w == target:
        continue
    vi = embeddings[w2i[w]]
    cos = (v @ vi) / (np.linalg.norm(v) * np.linalg.norm(vi) + 1e-9)
    sims.append((w, float(cos)))
sims.sort(key=lambda x: -x[1])

print(f"\nMost similar words to '{target}':")
for w, s in sims[:10]:
    print(f"  {w:15} -> {s:.4f}")


Vocabulary size: 68, Total tokens: 148
Epoch 1/15 - Loss: 17.1963
Epoch 2/15 - Loss: 14.8073
Epoch 3/15 - Loss: 13.8230
Epoch 4/15 - Loss: 13.3168
Epoch 5/15 - Loss: 12.9959
Epoch 6/15 - Loss: 12.7591
Epoch 7/15 - Loss: 12.5674
Epoch 8/15 - Loss: 12.4035
Epoch 9/15 - Loss: 12.2588
Epoch 10/15 - Loss: 12.1286
Epoch 11/15 - Loss: 12.0100
Epoch 12/15 - Loss: 11.9010
Epoch 13/15 - Loss: 11.8001
Epoch 14/15 - Loss: 11.7063
Epoch 15/15 - Loss: 11.6185

Most similar words to 'king':
  the             -> 0.8883
  queen           -> 0.6149
  kingdom         -> 0.4324
  information     -> 0.4172
  research        -> 0.4084
  is              -> 0.3937
  man             -> 0.3022
  woman           -> 0.2858
  learn           -> 0.2299
  and             -> 0.2113


### (b) Using Pretrained Model (Gensim)

In [3]:
%pip install gensim


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [4]:

import gensim.downloader as api

# Load pretrained GloVe vectors
model = api.load("glove-twitter-25")

target = "king"
print(f"Most similar words to '{target}' (pretrained GloVe):")
for word, sim in model.most_similar(target, topn=10):
    print(f"  {word:15} -> {sim:.4f}")


Most similar words to 'king' (pretrained GloVe):
  prince          -> 0.9337
  queen           -> 0.9202
  aka             -> 0.9177
  lady            -> 0.9163
  jack            -> 0.9147
  's              -> 0.9067
  stone           -> 0.8982
  mr.             -> 0.8919
  the             -> 0.8893
  star            -> 0.8892


## 6.2 Statistical N-gram Language Model (from scratch)
Predict the next word in a sequence.

In [5]:

corpus = "natural language processing is a branch of artificial intelligence.natural language processing enables computers to understand human language.machine learning is a branch of artificial intelligence.deep learning is a part of machine learning.language models learn patterns from text data.a language model predicts the next word in a sentence.natural language processing uses language models.artificial intelligence includes machine learning and deep learning."

# Sentence tokenization
sents = [s.strip() for s in corpus.split('.') if s.strip()]
toks = [s.lower().split() for s in sents]

from collections import Counter

unigram = Counter()
bigram = Counter()
trigram = Counter()

for s in toks:
    words = ['<s>'] + s + ['</s>']
    for i, w in enumerate(words):
        unigram[w] += 1
        if i > 0:
            bigram[(words[i-1], w)] += 1
        if i > 1:
            trigram[(words[i-2], words[i-1], w)] += 1

V = len(unigram)

def predict_bigram(prev, topk=5):
    cands = [(w, bigram[(prev, w)] + 1) for w in unigram]
    cands.sort(key=lambda x: -x[1])
    total = sum(c for _, c in cands)
    return [(w, c / total) for w, c in cands[:topk]]

def predict_trigram(w1, w2, topk=5):
    cands = [(w, trigram[(w1, w2, w)] + 1) for w in unigram]
    cands.sort(key=lambda x: -x[1])
    total = sum(c for _, c in cands)
    return [(w, c / total) for w, c in cands[:topk]]

print("Bigram predictions after 'natural':")
for w, p in predict_bigram('natural'):
    print(f"  {w:15} -> {p:.4f}")

print("\nTrigram predictions after 'natural language':")
for w, p in predict_trigram('natural', 'language'):
    print(f"  {w:15} -> {p:.4f}")

print("\nTrigram predictions after 'machine':")
# need two context words; use 'machine learning' if possible
print(predict_trigram('machine', 'learning') if False else "N/A")


Bigram predictions after 'natural':
  language        -> 0.1026
  <s>             -> 0.0256
  natural         -> 0.0256
  processing      -> 0.0256
  is              -> 0.0256

Trigram predictions after 'natural language':
  processing      -> 0.1026
  <s>             -> 0.0256
  natural         -> 0.0256
  language        -> 0.0256
  is              -> 0.0256

Trigram predictions after 'machine':
N/A
